# MolmoPoint-Vid-4B - VQA + pointing on sim frames
Kaggle, GPU T4 x2. Loads `allenai/MolmoPoint-Vid-4B` (Qwen3-4B + SigLIP, `trust_remote_code`, transformers 4.57.1 per the model card) and asks multiple-choice / yes-no questions built from the same cases as the EmbeddingGemma 2 notebook, then asks it to point to the door handle.

Notes:
- The checkpoint is stored in float32 (4.86B params, ~19.5 GB). T4 has no bfloat16 (the card uses bf16 autocast), so the default here is **fp32 split over both T4s** with `device_map='auto'`. `DTYPE='fp16'` (~9.7 GB) is faster but untested by Ai2 and may overflow.
- T4 is sm75: no FlashAttention 2, so `attn_implementation='sdpa'`.
- This checkpoint is trained for *video pointing* only; `allenai/MolmoPoint-8B` is the generalist (set `MODEL_ID`, needs fp16 or ~35 GB fp32 - does not fit 2xT4 in fp32).

Generated by `make_molmopoint_notebook.py`.

In [ ]:
!pip install -q transformers==4.57.1 accelerate einops decord2

In [ ]:
import torch, transformers
print('transformers', transformers.__version__)
print('torch', torch.__version__)
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'cuda:{i}', p.name, f'{p.total_memory/2**30:.1f} GB', 'bf16:', torch.cuda.is_bf16_supported())

Optional: if the HF download needs auth, add a Kaggle secret `HF_TOKEN`.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(UserSecretsClient().get_secret('HF_TOKEN'))
    print('HF login ok')
except Exception as e:
    print('No HF_TOKEN secret (fine for public models):', type(e).__name__)

## Images
Downloaded from imgbb with `wget`; images with no URL are decoded from the embedded base64.

In [ ]:
import base64, os, subprocess
IMAGES = [
 {
  "name": "wrist_2_jaws_from_2_sides_of_lever_almost_grasp.png",
  "url": "https://i.ibb.co/W4DShyjR/wrist-2-jaws-from-2-sides-of-lever-almost-grasp.png",
  "texts": [
   "robot jaws grasp door handle"
  ],
  "expected": [],
  "b64": ""
 },
 {
  "name": "dark_gray_pillar_blocks_door.png",
  "url": "https://i.ibb.co/V0sQPBXC/aec60258157e.png",
  "texts": [
   "door handle is hidden",
   "door handle is visible"
  ],
  "expected": [
   "door handle is hidden"
  ],
  "b64": ""
 },
 {
  "name": "latched_slip_door_closed.png",
  "url": "https://i.ibb.co/XrZ11SQr/final-head.png",
  "texts": [
   "door is closed",
   "door is open"
  ],
  "expected": [
   "door is closed"
  ],
  "b64": ""
 },
 {
  "name": "door_slight_open.png",
  "url": "https://i.ibb.co/jvMvHxwC/door-slight-open.png",
  "texts": [
   "door is closed",
   "door is slightly open",
   "door is a little open",
   "door is fully open"
  ],
  "expected": [
   "door is slightly open",
   "door is a little open"
  ],
  "b64": ""
 },
 {
  "name": "slip_mid_arc_not_grasping.png",
  "url": "https://i.ibb.co/4RctMj5D/slip-mid-arc-not-grasping.png",
  "texts": [
   "the robot arm grasps the door handle",
   "the robot arm does not grasp the door handle"
  ],
  "expected": [
   "the robot arm does not grasp the door handle"
  ],
  "b64": ""
 }
]

os.makedirs('imgs', exist_ok=True)
for im in IMAGES:
    im['path'] = os.path.join('imgs', im['name'])
    if im['url']:
        subprocess.run(['wget', '-q', '-O', im['path'], im['url']], check=True)
        src = 'wget ' + im['url']
    else:
        open(im['path'], 'wb').write(base64.b64decode(im['b64']))
        src = 'embedded base64'
    print(f"{im['name']}: {os.path.getsize(im['path'])} bytes via {src}")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1, len(IMAGES), figsize=(4 * len(IMAGES), 4))
for ax, im in zip(axs, IMAGES):
    ax.imshow(Image.open(im['path']).convert('RGB')); ax.set_title(im['name'], fontsize=7); ax.axis('off')
plt.show()

## Shared scoring helpers

In [ ]:
def report(name, texts, expected, scores, fmt="{:+.4f}"):
    """Print texts ranked by score; return PASS bool if `expected` is non-empty, else None."""
    print(f"\n{name}")
    for t, s in sorted(zip(texts, scores), key=lambda x: -x[1]):
        print(f"  {fmt.format(s)}  {t}")
    if not expected:
        return None
    good = max(s for t, s in zip(texts, scores) if t in expected)
    bad = max(s for t, s in zip(texts, scores) if t not in expected)
    ok = good > bad
    print(f"  expected best in {expected} -> {'PASS' if ok else 'FAIL'} (margin {good - bad:+.4f})")
    return ok


def summarize(results):
    scored = {k: v for k, v in results.items() if v is not None}
    print(f"\n[summary] {sum(scored.values())}/{len(scored)} PASS: "
          + ", ".join(f"{k}={'PASS' if v else 'FAIL'}" for k, v in scored.items()))


## VQA helpers (from common.py)

In [ ]:
def vqa_question(texts):
    """VQA prompt for one case: lettered multiple choice over `texts`, or yes/no for a single text.

    Returns (prompt, labels); labels are the answer tokens ('A', 'B', ... or 'yes', 'no').
    """
    if len(texts) == 1:
        return (f"Is this statement true for the image: '{texts[0]}'? Answer yes or no.", ["yes", "no"])
    labels = [chr(ord("A") + i) for i in range(len(texts))]
    opts = "\n".join(f"{l}. {t}" for l, t in zip(labels, texts))
    return ("Which statement best describes the image?\n" + opts
            + "\nAnswer with the letter only.", labels)


def parse_vqa_answer(answer, texts):
    """Map a free-text model answer to (label, chosen_text); (None, None) if unparseable.

    For single-text (yes/no) cases chosen_text is the text if the answer is 'yes', else None.
    """
    import re
    a = answer.strip()
    if len(texts) == 1:
        m = re.search(r"\b(yes|no)\b", a, re.I)
        if not m:
            return None, None
        label = m.group(1).lower()
        return label, (texts[0] if label == "yes" else None)
    labels = [chr(ord("A") + i) for i in range(len(texts))]
    pat = "|".join(labels)
    m = (re.match(rf"^\W*({pat})(?:\W|$)", a)
         or re.search(rf"(?:answer|option)\s*(?:is|:)?\s*\(?({pat})\b", a, re.I)
         or re.search(rf"\b({pat})[.)]", a))
    if m:
        label = m.group(1).upper()
        return label, texts[labels.index(label)]
    low = a.lower()
    hits = [t for t in sorted(texts, key=len, reverse=True) if t.lower() in low]
    if hits:
        return labels[texts.index(hits[0])], hits[0]
    return None, None


def label_token_ids(tokenizer, labels):
    """{label: [single-token ids of its variants ('A', ' A', 'a', ...)]} for logprob scoring."""
    out = {}
    for l in labels:
        ids = set()
        for v in {l, " " + l, l.lower(), " " + l.lower(), l.capitalize(), " " + l.capitalize()}:
            enc = tokenizer.encode(v, add_special_tokens=False)
            if len(enc) == 1:
                ids.add(enc[0])
        out[l] = sorted(ids)
    return out


def option_probs(first_logits, label_ids):
    """Softmax over the options of the first generated token's logits ({label: prob}), or None.

    Returns None when the greedy first token is not one of the option tokens (e.g. a preamble),
    since the first-step distribution then says little about the final choice.
    """
    import torch
    lp = torch.log_softmax(first_logits.float().reshape(-1), dim=-1)
    if not any(int(lp.argmax()) in ids for ids in label_ids.values()):
        return None
    per = {l: torch.logsumexp(lp[ids], 0) if ids else torch.tensor(-1e9) for l, ids in label_ids.items()}
    p = torch.softmax(torch.stack(list(per.values())), 0).tolist()
    return dict(zip(per, p))


def vqa_scores(texts, label, probs=None):
    """Per-text scores for report(): option probabilities if available, else 1/0 for the parsed choice.

    Single-text (yes/no) cases get one score: P(yes), or 1/0 for a yes/no answer.
    """
    if len(texts) == 1:
        if probs:
            return [probs["yes"]]
        return [1.0 if label == "yes" else 0.0]
    labels = [chr(ord("A") + i) for i in range(len(texts))]
    if probs:
        return [probs[l] for l in labels]
    return [1.0 if l == label else 0.0 for l in labels]


## Load model

In [ ]:
import time
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = 'allenai/MolmoPoint-Vid-4B'
DTYPE = 'fp32'  # 'fp32' (2 GPUs, safe) or 'fp16' (1 GPU, faster, may overflow); T4 has no bf16
dtype = torch.float32 if DTYPE == 'fp32' else torch.float16
n_gpu = torch.cuda.device_count()
max_memory = {i: '14GiB' for i in range(n_gpu)} if DTYPE == 'fp32' else None

t0 = time.time()
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, trust_remote_code=True, dtype=dtype,
    device_map='auto', max_memory=max_memory,
    attn_implementation='sdpa',  # T4 (sm75): no FlashAttention 2
)
model.eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, padding_side='left')
n = sum(p.numel() for p in model.parameters())
print(f'loaded {MODEL_ID} in {time.time()-t0:.1f}s, {n/1e9:.2f}B params, dtype {dtype}')
for i in range(n_gpu):
    print(f'  cuda:{i}: {torch.cuda.memory_allocated(i)/2**30:.2f} GB allocated')
print('device map:', sorted(set(map(str, getattr(model, 'hf_device_map', {}).values()))))

## Generation helper
Greedy decoding with the MolmoPoint logit processor (keeps point tokens valid). `output_logits=True` returns the raw first-step logits used for option probabilities.

In [ ]:
def run(image_path, prompt, max_new_tokens=32, pointing=False):
    msgs = [{'role': 'user', 'content': [
        {'type': 'text', 'text': prompt},
        {'type': 'image', 'image': Image.open(image_path).convert('RGB')}]}]
    inputs = processor.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt',
        return_dict=True, padding=True, return_pointing_metadata=pointing)
    metadata = inputs.pop('metadata', None)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.inference_mode():
        out = model.generate(**inputs, logits_processor=model.build_logit_processor_from_inputs(inputs),
                             max_new_tokens=max_new_tokens, do_sample=False,
                             output_logits=True, return_dict_in_generate=True)
    gen = out.sequences[:, inputs['input_ids'].size(1):]
    text = processor.post_process_image_text_to_text(
        gen, skip_special_tokens=not pointing, clean_up_tokenization_spaces=False)[0]
    return text, out.logits[0][0], metadata

## VQA per image
Multiple-choice cases list the candidate texts as lettered options (answer = letter); single-text cases ask yes/no. Scores passed to `report()` are the option probabilities from the first generated token when it is an option token, else 1/0 for the parsed answer.

In [ ]:
results = {}
for im in IMAGES:
    prompt, labels = vqa_question(im['texts'])
    t0 = time.time()
    answer, first_logits, _ = run(im['path'], prompt)
    label, chosen = parse_vqa_answer(answer, im['texts'])
    probs = option_probs(first_logits, label_token_ids(processor.tokenizer, labels))
    print(f"\n=== {im['name']} ({time.time()-t0:.1f}s)\n{prompt}")
    print(f'raw answer: {answer!r} -> label {label!r}, chosen {chosen!r}, probs {probs}')
    if label is None and probs is None:
        print('  could not parse the answer; scoring all options 0')
    results[im['name']] = report(im['name'], im['texts'], im['expected'],
                                 vqa_scores(im['texts'], label, probs))
summarize(results)

## Optional: point to the door handle
MolmoPoint emits special `<POINT_..>` tokens; `model.extract_image_points` decodes them with the processor's pointing metadata into `(object_id, image_num, x, y)` in original image pixels.

In [ ]:
import numpy as np
POINT_PROMPT = 'Point to the door handle'
fig, axs = plt.subplots(1, len(IMAGES), figsize=(4 * len(IMAGES), 4))
for ax, im in zip(axs, IMAGES):
    ax.imshow(Image.open(im['path']).convert('RGB')); ax.axis('off')
    try:
        text, _, md_ = run(im['path'], POINT_PROMPT, max_new_tokens=200, pointing=True)
    except Exception as e:
        print(f"{im['name']}: generation failed: {type(e).__name__}: {e}")
        ax.set_title(im['name'] + '\n(error)', fontsize=7); continue
    try:
        pts = np.array(model.extract_image_points(
            text, md_['token_pooling'], md_['subpatch_mapping'], md_['image_sizes']), dtype=float).reshape(-1, 4)
    except Exception as e:
        print(f"{im['name']}: point parse failed ({type(e).__name__}: {e}); raw: {text!r}")
        pts = np.zeros((0, 4))
    print(f"{im['name']}: {len(pts)} point(s) {pts[:, 2:].round(1).tolist()}")
    if not len(pts):
        print(f'  raw: {text!r}')
    ax.scatter(pts[:, 2], pts[:, 3], s=80, c='red', marker='x')
    ax.set_title(f"{im['name']}\n{len(pts)} point(s)", fontsize=7)
plt.show()